# Hypothesis Testing: Lab

### Real-world brief: is the quality difference real, or just noise?

A manufacturing plant collects tensile-strength and quality data on **1,800 parts** across three machines, two suppliers and two shifts. Managers keep asking *'is this difference real?'* — does a supplier really make stronger parts, does one machine lag, does the night shift fail more? In this lab you'll answer those questions properly with **hypothesis tests**: one/two-sample and paired **t-tests**, **one-way ANOVA**, and the **chi-square** test — always stating H₀/H₁, reading the **p-value** honestly, and reporting effect size.

**Resources provided:** `process_measurements.csv` (1,800 parts) and `calibration_trial.csv` (before/after a calibration). Built with scipy + pandas.

_Statistical foundations._

#objectives

State null and alternate hypotheses and pick a significance level

Run and interpret one-sample, two-sample and paired t-tests

Compare 3+ group means with one-way ANOVA

Test categorical association with the chi-square test

Read p-values correctly and report effect sizes, not just significance

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter. **Tip:** a column named `shift` would clash with pandas' `.shift()` method, so it's called `work_shift` — always use bracket access for safety, e.g. `df['work_shift']`.

In [ ]:
# === SETUP: build the data if missing ===
import os
import numpy as np
import pandas as pd


def build_quality(meas_path="process_measurements.csv", cal_path="calibration_trial.csv",
                  seed=271, verbose=False):
    """Manufacturing QC data for the hypothesis-testing lab. Effects are deliberately built in
    so each test has something real to find:

      process_measurements.csv  tensile strength + QC outcome per part, across machines,
                                suppliers and shifts (for one/two-sample t-tests, ANOVA, chi-square)
      calibration_trial.csv     strength before vs after a calibration, same machines (paired t-test)

    Built-in truths:
      - overall mean tensile > 500 MPa spec        -> one-sample t-test rejects
      - supplier X < supplier Y                    -> two-sample t-test rejects
      - machine A ~ B, but C clearly higher        -> ANOVA rejects (C is the odd one)
      - night shift fails more than day            -> chi-square (shift x pass) rejects
      - calibration raises strength                -> paired t-test rejects
    """
    rng = np.random.default_rng(seed)

    n_per = 300
    rows = []
    machine_base = {"A": 500.0, "B": 502.0, "C": 518.0}      # A~B, C higher
    supplier_off = {"X": -4.0, "Y": +4.0}                    # X lower, Y higher
    for machine in ["A", "B", "C"]:
        for supplier in ["X", "Y"]:
            k = n_per
            shift = rng.choice(["day", "night"], k, p=[0.5, 0.5])
            tensile = (machine_base[machine] + supplier_off[supplier]
                       + rng.normal(0, 12, k))
            roughness = np.clip(rng.normal(1.6, 0.4, k), 0.3, None)
            # fail probability: low tensile, rough surface, and night shift raise it
            logit = (-0.06 * (tensile - 500) + 1.4 * (roughness - 1.6)
                     + 0.7 * (shift == "night") - 1.0)
            pfail = 1 / (1 + np.exp(-logit))
            failed = rng.random(k) < pfail
            passed = np.where(failed, "fail", "pass")
            # defect type associated with supplier (X->porosity heavy, Y->crack heavy)
            dtype = []
            for f, sup in zip(failed, [supplier] * k):
                if not f:
                    dtype.append("none")
                else:
                    if sup == "X":
                        dtype.append(rng.choice(["porosity", "crack", "inclusion"], p=[0.6, 0.25, 0.15]))
                    else:
                        dtype.append(rng.choice(["porosity", "crack", "inclusion"], p=[0.25, 0.6, 0.15]))
            for i in range(k):
                rows.append((machine, supplier, shift[i], round(float(tensile[i]), 1),
                             round(float(roughness[i]), 2), passed[i], dtype[i]))
    meas = pd.DataFrame(rows, columns=["machine", "supplier", "work_shift", "tensile_strength_mpa",
                                       "surface_roughness_um", "passed", "defect_type"])
    meas.insert(0, "sample_id", [f"S{i:05d}" for i in range(len(meas))])
    meas = meas.sample(frac=1, random_state=seed).reset_index(drop=True)
    meas.to_csv(meas_path, index=False)

    # paired calibration trial: 30 machines measured before & after a calibration
    nm = 30
    before = rng.normal(498, 10, nm)
    after = before + rng.normal(5.0, 4.0, nm)        # calibration adds ~5 MPa
    cal = pd.DataFrame({
        "machine_id": [f"M{i:02d}" for i in range(1, nm + 1)],
        "strength_before": before.round(1), "strength_after": after.round(1),
    })
    cal.to_csv(cal_path, index=False)

    if verbose:
        print("measurements:", meas.shape, "| calibration:", cal.shape)
        print("mean tensile by machine:", meas.groupby("machine").tensile_strength_mpa.mean().round(1).to_dict())
        print("mean tensile by supplier:", meas.groupby("supplier").tensile_strength_mpa.mean().round(1).to_dict())
        print("fail rate by work_shift:", meas.assign(f=meas.passed.eq("fail")).groupby("work_shift").f.mean().round(3).to_dict())
        print("calibration before/after:", round(cal.strength_before.mean(), 1), "->", round(cal.strength_after.mean(), 1))
    return meas, cal

if not (os.path.exists('process_measurements.csv') and os.path.exists('calibration_trial.csv')):
    build_quality(); print('Generated data files.')
else:
    print('Found the provided data files.')

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
ALPHA = 0.05                       # our significance level, fixed in advance
df = pd.read_csv('process_measurements.csv')
cal = pd.read_csv('calibration_trial.csv')
print('measurements:', df.shape, '| calibration:', cal.shape)
df.head(3)

In [ ]:
# A tiny helper to report a test result consistently
def report(name, stat, p, h0, h1):
    print(f'{name}')
    print(f'  H0: {h0}')
    print(f'  H1: {h1}')
    print(f'  statistic = {stat:.3f}   p-value = {p:.3e}')
    verdict = 'REJECT H0 (significant)' if p <= ALPHA else 'fail to reject H0 (not significant)'
    print(f'  p {"<=" if p <= ALPHA else ">"} alpha={ALPHA}  ->  {verdict}\n')

#1. The t-test — comparing means

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. ONE-SAMPLE t-test: is mean tensile strength above the 500 MPa spec?
# -----------------------------------------------------------
x = df['tensile_strength_mpa']
t, p = stats.ttest_1samp(x, 500)
report('One-sample t-test vs 500 MPa', t, p,
       'mean tensile = 500 (process just meets spec)',
       'mean tensile != 500')
print(f'observed mean = {x.mean():.2f} MPa over n={len(x)} parts')

In [ ]:
# -----------------------------------------------------------
# 🔹 1B. TWO-SAMPLE t-test: do suppliers X and Y differ in strength?
# Good practice: check the assumptions first.
# -----------------------------------------------------------
xs = df[df.supplier == 'X']['tensile_strength_mpa']
ys = df[df.supplier == 'Y']['tensile_strength_mpa']
# assumption checks: normality (Shapiro on a sample) and equal variance (Levene)
print('Levene equal-variance p =', round(stats.levene(xs, ys).pvalue, 3))
t, p = stats.ttest_ind(xs, ys, equal_var=True)
report('Two-sample t-test: supplier X vs Y', t, p,
       'mean(X) = mean(Y)', 'mean(X) != mean(Y)')
# effect size (Cohen's d) — is the difference practically meaningful?
pooled = np.sqrt(((len(xs)-1)*xs.var() + (len(ys)-1)*ys.var()) / (len(xs)+len(ys)-2))
d = (ys.mean() - xs.mean()) / pooled
print(f"means: X={xs.mean():.1f}, Y={ys.mean():.1f} | Cohen's d = {d:.2f} (effect size)")

In [ ]:
# -----------------------------------------------------------
# 🔹 1C. PAIRED t-test: did a calibration raise strength on the SAME machines?
# -----------------------------------------------------------
t, p = stats.ttest_rel(cal['strength_after'], cal['strength_before'])
report('Paired t-test: after vs before calibration', t, p,
       'no change (mean difference = 0)', 'calibration changed the strength')
diff = cal['strength_after'] - cal['strength_before']
print(f'mean improvement = {diff.mean():.2f} MPa across {len(cal)} machines')
print('Pairing removes machine-to-machine variation, making the test more sensitive.')

#### 🧪 EXERCISE 1 — Compare two machines
1. Run a two-sample t-test comparing **machine A vs machine B** tensile strength. State H₀/H₁ (use the `report` helper) and interpret the p-value.
2. You'll likely find it is *not* significant. In a comment, explain what 'fail to reject H₀' does and does **not** let you conclude (hint: it isn't proof the machines are identical).

In [ ]:
# 1. two-sample t-test: machine A vs machine B
# YOUR CODE HERE

# 2. what 'fail to reject' means: ...   (comment)

#2. ANOVA — comparing 3+ means

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. ONE-WAY ANOVA: do machines A, B, C have equal mean strength?
# (Doing three t-tests instead would inflate the false-positive rate.)
# -----------------------------------------------------------
groups = [df[df.machine == m]['tensile_strength_mpa'] for m in ['A', 'B', 'C']]
f, p = stats.f_oneway(*groups)
report('One-way ANOVA across machines A/B/C', f, p,
       'all machine means are equal', 'at least one machine differs')
print('group means:', {m: round(g.mean(), 1) for m, g in zip(['A', 'B', 'C'], groups)})

In [ ]:
# Visualise the group distributions
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.boxplot([g.values for g in groups], labels=['A', 'B', 'C'])
ax.set_xlabel('machine'); ax.set_ylabel('tensile strength (MPa)')
ax.axhline(500, color='#C0392B', ls='--', lw=1, label='spec 500')
ax.set_title('Tensile strength by machine'); ax.legend(); plt.tight_layout(); plt.show()
print('ANOVA says they are not all equal; the boxplot shows machine C is the standout.')

#### 🧪 EXERCISE 2 — Which machine differs, and by how much?
1. ANOVA only says *at least one* group differs. Run pairwise two-sample t-tests (A-B, A-C, B-C) to see **which** pairs differ. (This is an informal post-hoc; a proper analysis would use Tukey's HSD.)
2. In a comment, explain why running many pairwise tests without correction inflates the chance of a false positive — and name one correction (e.g. Bonferroni).

In [ ]:
# 1. pairwise t-tests A-B, A-C, B-C
# YOUR CODE HERE

# 2. multiple-comparisons problem + a correction: ...   (comment)

#3. The chi-square test — categorical association

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. CHI-SQUARE: is the pass/fail outcome associated with the shift?
# -----------------------------------------------------------
ct = pd.crosstab(df['work_shift'], df['passed'])      # contingency table of counts
print('observed counts:'); print(ct, '\n')
chi2, p, dof, expected = stats.chi2_contingency(ct)
report('Chi-square: work_shift vs pass/fail', chi2, p,
       'shift and outcome are independent', 'they are associated')
fail_rate = (ct['fail'] / ct.sum(axis=1)).round(3)
print('fail rate by shift:', fail_rate.to_dict())

In [ ]:
# Observed vs expected makes the association concrete
exp = pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(1)
print('expected counts IF shift and outcome were independent:'); print(exp)
print('\nNight has MORE failures than independence would predict -> the variables are linked.')

#### 🧪 EXERCISE 3 — Supplier vs defect type
1. Among **failed** parts only, build a contingency table of `supplier` × `defect_type` and run a chi-square test. Is the type of defect associated with the supplier?
2. In a comment, interpret the result for the plant: if significant, what might it suggest about each supplier's process?

In [ ]:
# 1. chi-square: supplier vs defect_type among failed parts
# YOUR CODE HERE

# 2. interpretation for the plant: ...   (comment)

#4. Choosing the right test & avoiding traps

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. A decision guide
# -----------------------------------------------------------
guide = pd.DataFrame([
    ['one numeric mean vs a target', 'one-sample t-test', 'ttest_1samp'],
    ['two independent group means', 'two-sample t-test', 'ttest_ind'],
    ['before vs after on same units', 'paired t-test', 'ttest_rel'],
    ['three or more group means', 'one-way ANOVA', 'f_oneway'],
    ['association of two categoricals', 'chi-square test', 'chi2_contingency'],
], columns=['your question', 'test', 'scipy function'])
print(guide.to_string(index=False))

In [ ]:
# -----------------------------------------------------------
# 🔹 4B. WHY 'significant' isn't always 'important' — and the danger of many tests
# -----------------------------------------------------------
# (i) huge n can make a trivial difference 'significant'
rng = np.random.default_rng(0)
a = rng.normal(100.0, 10, 100000); b = rng.normal(100.1, 10, 100000)   # 0.1 unit difference
t, p = stats.ttest_ind(a, b)
print(f'trivial 0.1-unit difference, n=100k each: p={p:.4f}  -> significant but meaningless')
print(f"  Cohen's d = {(b.mean()-a.mean())/10:.3f} (tiny effect)\n")
# (ii) multiple comparisons: test 20 pure-noise pairs at alpha=0.05
false_pos = sum(stats.ttest_ind(rng.normal(0,1,200), rng.normal(0,1,200)).pvalue < 0.05 for _ in range(20))
print(f'20 tests on pure noise at alpha=0.05 -> {false_pos} "significant" by chance (expected ~1)')
print('Lesson: report effect size, and correct for multiple comparisons.')

#### 🧪 EXERCISE 4 — Put it all together
A plant engineer asks three questions. For each, name the **correct test**, then run it on this data and give a one-line conclusion:
1. *'Is the night shift's roughness different from the day shift's?'*
2. *'Do our three machines differ in surface roughness?'*
3. *'Is passing/failing related to the supplier?'*

In [ ]:
# 1-3. pick and run the right test for each question
# YOUR CODE HERE

#📘 Summary

| Question | Test | Result on this data |
| -------- | ---- | ------------------- |
| mean strength vs 500 spec | one-sample t | significantly above 500 |
| supplier X vs Y strength | two-sample t | Y significantly stronger |
| before vs after calibration | paired t | significant improvement |
| machines A/B/C strength | ANOVA | not all equal (C differs) |
| shift vs pass/fail | chi-square | associated (night fails more) |

**Core lesson:** hypothesis testing turns 'looks different' into a defensible decision. State H₀/H₁ and α up front, pick the test from your data's shape (numeric vs categorical, how many groups), read the p-value as evidence against H₀ — and always check assumptions and report the **effect size**, because statistical significance is not the same as practical importance.